# **FAERS - Data Processing & Feature Engineering**

# Load Data

In [ ]:
%pip install dask[dataframe] adlfs pyarrow

In [ ]:
import dask.dataframe as dd
import pandas as pd
from collections import Counter
import numpy as np
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

In [ ]:
df = dd.read_parquet('FAERS_final.parquet')

print(df.npartitions)
#df.head()

# Missing values

In [ ]:
df['sex_bin'] = df['sex_bin'].fillna("Unknown")
df['wt_kg'] = df['wt_kg'].fillna(-1)
df['age_years'] = df['age_years'].fillna(-1)

# Flatten

In [ ]:
def flatten_to_strings(x):
    out = []
    if x is None or (isinstance(x, float) and pd.isna(x)) or x is pd.NA:
        return out
    
    # define terms that don't count as real medical info
    exclude_terms = ["unknown", "nan", "", "no adverse event", "none", "product quality issue"]
    
    if isinstance(x, (str, int, float)):
        s = str(x).strip()
        if s.lower() not in exclude_terms:
            out.append(s)
        return out
    
    try:
        for item in x:
            out.extend(flatten_to_strings(item))
    except TypeError:
        pass
    return list(set(out))

df['rx_flat'] = df['rx_name'].map(flatten_to_strings, meta=('rx_flat','object'))
df['reacs_flat'] = df['reactions'].map(flatten_to_strings, meta=('reacs_flat','object'))

print("Flattened lists done")


# Target col (outcomes)

In [ ]:
# target lable
outcome_cols = ['outcome_other_serious', 'outcome_intervention_required', 'outcome_congenital_anomaly', 'outcome_disability', 'outcome_hospitalisation', 'outcome_life_threatening', 'outcome_death']
# none of the above = 0, outcome_other_serious = 1, outcome_intervention_required = 2, outcome_congenital_anomaly = 3, outcome_disability = 4, outcome_hospitalisation = 5, outcome_life_threatening = 6, outcome_death = 7
def get_single_label(partition_df):
    target = pd.Series(0, index=partition_df.index, dtype='int8')
    existing_outcomes = [c for c in outcome_cols if c in partition_df.columns]
    
    for i, col in enumerate(outcome_cols, 1):
        if col in existing_outcomes:
            target.loc[partition_df[col] == 1] = i
    return target

# map partition
df['target_label'] = df.map_partitions(get_single_label, meta=('target_label', 'int8'))

# Unknown cols

In [ ]:
# if flattening resulted in an empty list then marked as "Unknown"
df['rx_unknown'] = df['rx_flat'].map(lambda x: 1 if len(x) == 0 else 0, meta=('rx_unknown', 'int8'))
df['reacs_unknown'] = df['reacs_flat'].map(lambda x: 1 if len(x) == 0 else 0, meta=('reacs_unknown', 'int8'))

# standard flags for scalars
df['sex_unknown'] = (df['sex_bin'] == "Unknown").astype('int8')
df['wt_unknown'] = (df['wt_kg'] == -1).astype('int8')
df['age_unknown'] = (df['age_years'] == -1).astype('int8')

print("Unknown flags created based on filtered results")

# Top N drugs and reactions

In [ ]:
def get_strictly_flat_top(series, top_n):
    data = series.compute()
    flat_pool = [str(item) for sublist in data for item in sublist]
    counts = Counter(flat_pool)
    return [k for k, _ in counts.most_common(top_n)]

top_550_drugs = get_strictly_flat_top(df['rx_flat'], 550)
top_300_reacs = get_strictly_flat_top(df['reacs_flat'], 300)

print("Top items calculation done")

# Multi-hot encoding

In [ ]:
top_rx_set = set(top_550_drugs)
top_reac_set = set(top_300_reacs)

def encode_binary(lst, top_list, top_set):
    s_set = set(lst)
    vec = [1 if x in s_set else 0 for x in top_list]
    other = 1 if len(s_set - top_set) > 0 else 0
    return vec, other

df['rx_mh'] = df['rx_flat'].map(lambda x: encode_binary(x, top_550_drugs, top_rx_set)[0], meta=('rx_mh', 'object'))
df['other_rx'] = df['rx_flat'].map(lambda x: encode_binary(x, top_550_drugs, top_rx_set)[1], meta=('other_rx', 'int8'))

df['reac_mh'] = df['reacs_flat'].map(lambda x: encode_binary(x, top_300_reacs, top_reac_set)[0], meta=('reac_mh', 'object'))
df['other_reacs'] = df['reacs_flat'].map(lambda x: encode_binary(x, top_300_reacs, top_reac_set)[1], meta=('other_reacs', 'int8'))

# Groupby caseid

In [ ]:
agg_dict = {
    'sex_bin': 'first', 'sex_unknown': 'max',
    'age_years': 'first', 'age_unknown': 'max',
    'wt_kg': 'first', 'wt_unknown': 'max',
    'rx_unknown': 'max', 'reacs_unknown': 'max',
    'other_rx': 'max', 'other_reacs': 'max',
    'rx_mh': 'max', 'reac_mh': 'max',
    'target_label': 'max'  
}

df_grouped = df.groupby("caseid").agg(agg_dict)

print("Grouped by caseid done")

# Expand multi-hot

In [ ]:
def expand_vectors(s, names):
    data = np.array(s.tolist())
    if data.ndim == 1:
        data = data.reshape(len(s), -1)
    return pd.DataFrame(data, index=s.index, columns=names).astype('int8')

rx_features = df_grouped['rx_mh'].map_partitions(expand_vectors, names=top_550_drugs, meta={n: 'int8' for n in top_550_drugs})

reac_features = df_grouped['reac_mh'].map_partitions(expand_vectors, names=top_300_reacs, meta={n: 'int8' for n in top_300_reacs})

# Final df

In [ ]:
df_final = dd.concat([df_grouped.drop(columns=['rx_mh', 'reac_mh']), rx_features, reac_features], axis=1).fillna(0)

print("final dataset compilation done")
#print(df_final.head(5))